**Table of contents**<a id='toc0_'></a>    
- [Fluopy trials notebook](#toc1_)    
  - [Importing all modules](#toc1_1_)    
  - [Routine template](#toc1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Fluopy trials notebook](#toc0_)
This notebook is dedicated to test functionality of fluopy

## <a id='toc1_1_'></a>[Importing all modules](#toc0_)

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import fluopy.analysis as an
import fluopy.blinking as bl
import fluopy.distributions as dist
import fluopy.emissions as em
import fluopy.fcs as fcs_p
import fluopy.figure as fi
import fluopy.fitting as fit
import fluopy.fluorophores as fl
import fluopy.formulas as fo
import fluopy.miscellaneous as mi
import fluopy.network as net
import fluopy.prediction as pr
import fluopy.simulation_tcspc as si_tcspc
import fluopy.simulation as si
import fluopy.transitions as tr

%load_ext autoreload
%autoreload 2


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

for package in [
    an,
    bl,
    dist,
    em,
    fcs_p,
    fi,
    fit,
    fl,
    fo,
    mi,
    net,
    pr,
    si_tcspc,
    si,
    tr,
]:
    print(f"{package.__name__} version: {package.__version__}")

fluopy.analysis version: 0.1.0
fluopy.blinking version: 0.1.0
fluopy.distributions version: 0.1.0
fluopy.emissions version: 0.1.0
fluopy.fcs version: 0.1.0
fluopy.figure version: 0.1.0
fluopy.fitting version: 0.1.0
fluopy.fluorophores version: 0.1.0
fluopy.formulas version: 0.1.0
fluopy.miscellaneous version: 0.1.0
fluopy.network version: 0.1.0
fluopy.prediction version: 0.1.0
fluopy.simulation_tcspc version: 0.1.0
fluopy.simulation version: 0.1.0
fluopy.transitions version: 0.1.0


## <a id='toc1_2_'></a>[Routine template](#toc0_)

In [52]:
fluorophores = fl.construct_fluorophores(
    name="cy5_dna", distance=3, count=4, shape="square"
)
fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=5,
    wavelength=640,
    bleaching=True,
    energy_transfer=True,
    dstorm=True,
    dstorm_parameters={"reducing_agent": "mea", "concentration": 100, "ph": 7.5},
    energy_transfer_parameters={"overwrite": {"off": [1, 0.00001]}},
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set = transition_set.adjust_rates({8: 5e1})
transition_set.finalize()

transition_set.transition_df

transition_type  \
Fluorophore                         identity                                           
cy5_dna                             0                      TransitionType.EXCITATION   
                                    1            TransitionType.FLUORESCENT_EMISSION   
                                    2         TransitionType.INTERSYSTEM_CROSSING_ST   
                                    3                   TransitionType.ISOMERIZATION   
                                    4                      TransitionType.THERM_BISO   
                                    5                     TransitionType.REDUCTION_T   
                                    6                     TransitionType.REDUCTION_S   
                                    7               TransitionType.THERM_ELIMINATION   
                                    8                TransitionType.PHOTOBLEACHING_1   
                                    9               TransitionType.S1_S0_TRANSITIONS   
                                    10              TransitionType.T1_S0_TRANSITIONS   
                                    11             TransitionType.CIS_S0_TRANSITIONS   
                                    12             TransitionType.OFF_S0_TRANSITIONS   
D: cy5_dna, A: cy5_dna, dist: 3.0   13                     TransitionType.CIS_FRET_1   
                                    14                     TransitionType.CIS_FRET_2   
                                    15                     TransitionType.OFF_FRET_1   
                                    16                     TransitionType.OFF_FRET_2   
                                    17                           TransitionType.FRET   
                                    18               TransitionType.S_S_ANNIHILATION   
                                    19             TransitionType.S_T_ANNIHILATION_1   
D: cy5_dna, A: cy5_dna, dist: 4.243 20                     TransitionType.CIS_FRET_1   
                                    21                     TransitionType.CIS_FRET_2   
                                    22                     TransitionType.OFF_FRET_1   
                                    23                     TransitionType.OFF_FRET_2   
                                    24                           TransitionType.FRET   
                                    25               TransitionType.S_S_ANNIHILATION   
                                    26             TransitionType.S_T_ANNIHILATION_1   

                                             abbreviation       initial_state  \
Fluorophore                         identity                                    
cy5_dna                             0                 EXC      SingleState.S0   
                                    1                 FLU      SingleState.S1   
                                    2               ISCST      SingleState.S1   
                                    3                 ISO      SingleState.S1   
                                    4               TBISO     SingleState.Cis   
                                    5                REDT      SingleState.T1   
                                    6                REDS      SingleState.S1   
                                    7                  TE     SingleState.OFF   
                                    8                 BLE      SingleState.T1   
                                    9             S1S0SUM      SingleState.S1   
                                    10            T1S0SUM      SingleState.T1   
                                    11           CisS0SUM     SingleState.Cis   
                                    12         OFF_S0_SUM     SingleState.OFF   
D: cy5_dna, A: cy5_dna, dist: 3.0   13             CFRET1  PairedState.S1_Cis   
                                    14             CFRET2  PairedState.S1_Cis   
                                    15             OFRET1  PairedState.S1_OFF   
                                    16             OFRET2 

In [3]:
sim = si.Simulation(transition_set)
sim.run(end_time=None)

WARNING for line:                 warnings.warn(
 Floating point precision error warning:
 The higher limit of smallest increment with a probability of 1.00e-03 is 1.69e-12.
 This was estimated using the highest possible rate which occurs for example in state combination [1].
 Everything drawn below this number will be rounded to zero starting somewhere between 1.00e+03 - 1.00e+04. 
